# Longitudinal Analysis: Dopamine Asymmetry and Psychosis

**Goal**: Test whether baseline dopaminergic brain asymmetry predicts psychosis severity at follow-up,
and whether changes in asymmetry track changes in symptoms.

- Cross-sectional replication at each follow-up timepoint
- Baseline brain → follow-up psychosis (cross-timepoint prediction)
- Change score analysis (delta-AI predicts delta-PQ-BC?)
- Worsening group analysis
- Test-retest reliability (ICC for AI features)

In [ ]:
from core.config import initialize_notebook, get_figures_dir

env = initialize_notebook(regenerate_run_id=False)

TARGET_COL = "pps_y_ss_severity_score"
TARGET_NAME = "pps_severity_raw"
NETWORK = "dopamine_core"

SEED = env.configs.run["seed"]
seed = SEED  # alias for cells that use lowercase
N_SPLITS = env.configs.regression.get("cv", {}).get("n_outer_splits", 5)
N_PERMS = env.configs.regression.get("permutation", {}).get("n_permutations", 1000)
N_BOOTSTRAP = env.configs.regression.get("bootstrap", {}).get("n_bootstrap", 10000)
CUTOFF = env.configs.regression.get("sample_weighting", {}).get("custom_bins", {}).get(
    TARGET_NAME, [30])[0]

feature_transform = env.configs.regression.get("feature_transform", "asymmetry")
fig_dir = get_figures_dir(env, "longitudinal")

print(f"Run: {env.configs.run['run_name']}")
print(f"Target: {TARGET_NAME} ({TARGET_COL})")
print(f"Network: {NETWORK}")
print(f"Feature transform: {feature_transform}")
print(f"CV folds: {N_SPLITS} | Seed: {SEED} | Cutoff: {CUTOFF}")

## Cell 1: Load Longitudinal Data

In [ ]:
from core.regression.longitudinal import load_longitudinal_data
from core.regression.univariate import extract_bilateral_pairs
from core.tsne.embeddings import get_roi_columns_from_config

long_df, wide_df, subject_ids = load_longitudinal_data(env, TARGET_COL)

# Get network features and bilateral pairs
roi_columns = get_roi_columns_from_config(env.configs.data, [NETWORK])
bilateral_pairs, unilateral = extract_bilateral_pairs(env.configs.data, [NETWORK])

timepoints = env.configs.data["timepoints"]
tp_col = env.configs.data["columns"]["mapping"]["timepoint"]

print(f"Longitudinal subjects (baseline + >=1 follow-up): {len(subject_ids)}")
print(f"\nSubjects per timepoint:")
for tp_name, tp_val in timepoints.items():
    n = long_df[long_df[tp_col] == tp_val].shape[0]
    print(f"  {tp_name}: {n}")
print(f"\nBilateral pairs: {len(bilateral_pairs)}")
print(f"ROI columns: {len(roi_columns)}")

## Cell 2: Descriptive Statistics

PQ-BC trajectories, attrition, demographics.

In [ ]:
from core.regression.visualization import plot_longitudinal_overview

# Collect timepoints with data
tp_with_data = []
for tp_name, tp_val in timepoints.items():
    n = long_df[long_df[tp_col] == tp_val][TARGET_COL].notna().sum()
    if n > 0:
        tp_with_data.append((tp_name, tp_val, n))

id_col = env.configs.data["columns"]["mapping"]["id"]
plot_longitudinal_overview(long_df, wide_df, tp_with_data, tp_col, TARGET_COL, id_col,
                            save_path=fig_dir / "longitudinal_overview.png")

## Cross-Sectional Replication Across Timepoints

Does the AI-PQ-BC correlation replicate at each timepoint? Raw vs Asymmetry comparison.

In [ ]:
from core.regression.longitudinal import stability_analysis
from core.regression.visualization import plot_stability_heatmaps

stability_raw = stability_analysis(long_df, env, bilateral_pairs, roi_columns, TARGET_COL, feature_transform="raw")
stability_ai  = stability_analysis(long_df, env, bilateral_pairs, roi_columns, TARGET_COL, feature_transform="asymmetry")

for lbl, sdf in [("RAW L/R", stability_raw), ("ASYMMETRY INDEX (AI)", stability_ai)]:
    hd = sdf[sdf["r"].notna()]
    print(f"\n{'='*70}\n  {lbl}\n{'='*70}")
    print(hd.to_string(index=False, float_format=lambda x: f"{x:.4f}") if len(hd) else "No data.")

plot_stability_heatmaps(stability_raw, stability_ai,
                         save_path=fig_dir / "stability_heatmaps.png")

## Attrition Analysis

Are subjects who drop out systematically different from completers?
Tests for selection bias in longitudinal sample.

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import ttest_ind, chi2_contingency
from core.regression.visualization import plot_attrition_results

id_col       = env.configs.data["columns"]["mapping"]["id"]
baseline_tp  = env.configs.data["timepoints"]["baseline"]
year2_tp     = env.configs.data["timepoints"]["year2"]
age_col      = env.configs.data["columns"]["mapping"]["age"]
sex_col      = env.configs.data["columns"]["mapping"]["sex"]

bl = long_df[long_df[tp_col] == baseline_tp].copy()
y2_ids = long_df[long_df[tp_col] == year2_tp][id_col].unique()
bl["has_year2"] = bl[id_col].isin(y2_ids)
n_comp = bl["has_year2"].sum(); n_drop = (~bl["has_year2"]).sum()
print(f"Completers: {n_comp}  |  Dropouts: {n_drop}  |  Retention: {100*n_comp/len(bl):.1f}%")

y_comp = bl.loc[bl["has_year2"],  TARGET_COL].dropna()
y_drop = bl.loc[~bl["has_year2"], TARGET_COL].dropna()
t, p = ttest_ind(y_comp, y_drop)
d = (y_comp.mean()-y_drop.mean()) / np.sqrt(((len(y_comp)-1)*y_comp.std()**2 + (len(y_drop)-1)*y_drop.std()**2)/(len(y_comp)+len(y_drop)-2))
print(f"PQ-BC: comp={y_comp.mean():.2f}, drop={y_drop.mean():.2f}, t={t:.2f}, p={p:.4f}, d={d:+.3f}")

present_cols = [c for c in roi_columns if c in bl.columns]
all_d = [d]; all_labels = ["PQ-BC"]
for col in present_cols:
    vc = bl.loc[bl["has_year2"],  col].dropna().astype(float)
    vd = bl.loc[~bl["has_year2"], col].dropna().astype(float)
    if len(vc)>10 and len(vd)>10:
        df_ = (vc.mean()-vd.mean())/np.sqrt(((len(vc)-1)*vc.std()**2+(len(vd)-1)*vd.std()**2)/(len(vc)+len(vd)-2))
        all_d.append(df_); all_labels.append(col.replace("smri_vol_scs_","").replace("dmri_dtimd_fiberat_","CST_"))

plot_attrition_results(y_comp, y_drop, d, p, all_d, all_labels, n_comp, n_drop,
                        save_path=fig_dir / "attrition.png")

## Developmental Change in Asymmetry

Paired t-tests: do AI features shift from baseline to year 2?
Tests whether dopaminergic asymmetry changes during development (ages 9-11).

In [ ]:
import numpy as np
from scipy.stats import ttest_rel
from core.regression.univariate import compute_asymmetry_features
from core.regression.longitudinal import get_paired_bl_y2
from core.regression.visualization import plot_developmental_change

id_col      = env.configs.data["columns"]["mapping"]["id"]
baseline_tp = env.configs.data["timepoints"]["baseline"]
year2_tp    = env.configs.data["timepoints"]["year2"]

bl_v, y2_v, X_bl, X_y2, asym_bl, asym_y2, valid_pairs, present_cols = get_paired_bl_y2(
    long_df, tp_col, baseline_tp, year2_tp, roi_columns, bilateral_pairs, id_col,
    harm_cfg=dict(env.configs.harmonize))
n_valid = len(bl_v)
ai_names = sorted(k for k in asym_bl if k.endswith("_AI"))

print(f"DEVELOPMENTAL CHANGE IN ASYMMETRY: BL vs Y2 (n={n_valid})")
print(f"{'Feature':<20} {'BL mean':>10} {'Y2 mean':>10} {'Delta':>10} {'t':>8} {'p':>10}")
print("-"*70)
results_dev = []
for name in ai_names:
    bv = asym_bl[name]; yv = asym_y2[name]; dlt = yv - bv
    t, p = ttest_rel(bv, yv); d = dlt.mean()/dlt.std() if dlt.std()>0 else 0
    sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else ""))
    print(f"{name:<20} {bv.mean():>+10.5f} {yv.mean():>+10.5f} {dlt.mean():>+10.5f} {t:>8.2f} {p:>10.4f} {sig}")
    results_dev.append({"feature":name,"bl_mean":bv.mean(),"y2_mean":yv.mean(),"delta_mean":dlt.mean(),"delta_std":dlt.std(),"t":t,"p":p,"d":d})

plot_developmental_change(results_dev, asym_bl, asym_y2, ai_names, n_valid,
                           save_path=fig_dir / "developmental_change.png")

## Severity × Developmental Trajectory Interaction

Does baseline PQ-BC severity predict the **rate** of pallidum AI change?
If high-symptom children show a different asymmetry trajectory, this supports
an aberrant neurodevelopmental model 

## Worsening Group Analysis

Do worseners (delta PQ-BC ≥ 0.5 SD) have different baseline features than stable subjects?
Compares raw volumes AND asymmetry indices.

In [ ]:
from core.regression.longitudinal import worsening_group_analysis
import matplotlib.pyplot as plt

if f"{TARGET_COL}_year2" in wide_df.columns:
    # Align worsening threshold: 0.5 × full SD (defensible; matches classifier in analyses below)
    bl_col_w = f"{TARGET_COL}_baseline"
    fu_col_w = f"{TARGET_COL}_year2"
    paired_w = wide_df.dropna(subset=[bl_col_w, fu_col_w])
    half_sd = 0.5 * paired_w[bl_col_w].std()
    mcid_half = np.ceil(half_sd)
    print(f"Worsening threshold: 0.5 SD = {half_sd:.1f} → {mcid_half:.0f} pts (n paired={len(paired_w)})")

    worsen_raw = worsening_group_analysis(
        wide_df, env, bilateral_pairs, roi_columns, TARGET_COL,
        baseline_name="baseline", followup_name="year2", feature_transform="raw",
        threshold=mcid_half,
    )
    worsen_ai = worsening_group_analysis(
        wide_df, env, bilateral_pairs, roi_columns, TARGET_COL,
        baseline_name="baseline", followup_name="year2", feature_transform="asymmetry",
        threshold=mcid_half,
    )

    for label, wdf in [("RAW L/R FEATURES", worsen_raw), ("ASYMMETRY INDEX (AI)", worsen_ai)]:
        print(f"\n{'='*80}")
        print(f"  WORSENING GROUP ANALYSIS: {label}")
        print(f"{'='*80}")
        print(wdf[["feature", "mean_worsened", "mean_stable", "t", "p", "p_fdr", "cohens_d",
                    "n_worsened", "n_stable"]].to_string(index=False))
        n_raw = (wdf["p"] < 0.05).sum()
        n_fdr = (wdf["p_fdr"] < 0.05).sum()

    # Side-by-side forest plots (colored by FDR)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for ax, label, wdf in [(ax1, "Raw L/R", worsen_raw), (ax2, "Asymmetry Index", worsen_ai)]:
        feats  = wdf["feature"].values
        d_vals = wdf["cohens_d"].values
        p_fdr  = wdf["p_fdr"].values
        short  = [f.replace("smri_vol_scs_", "").replace("dmri_dtimd_fiberat_", "CST ")
                   .replace("_AI", "") for f in feats]
        sorted_idx = np.argsort(d_vals)
        colors = ["crimson" if p_fdr[i] < 0.05 else "steelblue" for i in sorted_idx]
        ax.barh(range(len(feats)), [d_vals[i] for i in sorted_idx],
                color=colors, alpha=0.8, edgecolor="black")
        ax.set_yticks(range(len(feats)))
        ax.set_yticklabels([short[i] for i in sorted_idx], fontsize=9)
        ax.set_xlabel("Cohen's d (Worsened − Stable)", fontweight="bold")
        n_w = int(wdf["n_worsened"].iloc[0])
        n_s = int(wdf["n_stable"].iloc[0])
        ax.set_title(f"{label}\nWorseners (n={n_w}) vs Stable (n={n_s})", fontweight="bold")
        ax.axvline(0, color="gray", ls="--", lw=0.8)
        ax.legend(handles=[
            plt.Rectangle((0,0),1,1, fc="crimson", ec="black", label="FDR p<0.05"),
            plt.Rectangle((0,0),1,1, fc="steelblue", ec="black", label="n.s."),
        ], loc="lower right", fontsize=8)

    plt.suptitle("Worsening Group: Raw Volumes vs Asymmetry", fontweight="bold", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Year 2 follow-up data not available")

## Test-Retest Reliability (ICC)

ICC(3,1) for AI and raw features: baseline vs year 2.
Quantifies measurement reliability and explains longitudinal sensitivity.

In [ ]:
import numpy as np, pandas as pd
import pingouin as pg
from core.regression.univariate import compute_asymmetry_features
from core.regression.longitudinal import get_paired_bl_y2
from core.regression.visualization import plot_icc_bars

id_col      = env.configs.data["columns"]["mapping"]["id"]
baseline_tp = env.configs.data["timepoints"]["baseline"]
year2_tp    = env.configs.data["timepoints"].get("year2")

if year2_tp is None:
    print("Year 2 not configured."); raise SystemExit

bl_v, y2_v, X_bl, X_y2, asym_bl, asym_y2, valid_pairs, present_cols = get_paired_bl_y2(
    long_df, tp_col, baseline_tp, year2_tp, roi_columns, bilateral_pairs, id_col,
    harm_cfg=dict(env.configs.harmonize))
ids_valid = bl_v[id_col].values
print(f"Shared subjects: {len(ids_valid)}")

def _icc(feat_name, bl_arr, y2_arr, ids):
    df_icc = pd.DataFrame({"targets":np.tile(ids,2),"raters":["bl"]*len(ids)+["y2"]*len(ids),"ratings":np.concatenate([bl_arr,y2_arr])})
    res = pg.intraclass_corr(data=df_icc,targets="targets",raters="raters",ratings="ratings")
    return res[res["Type"]=="ICC3"].iloc[0]

ai_rows, raw_rows = [], []
print(f"\n{'='*60}\n  ASYMMETRY INDEX (AI) — ICC(3,1)\n{'='*60}")
for name in sorted(k for k in asym_bl if k.endswith("_AI")):
    r = _icc(name, asym_bl[name], asym_y2[name], ids_valid)
    print(f"{name:<20} {r['ICC']:>8.3f} [{r['CI95%'][0]:.3f},{r['CI95%'][1]:.3f}] {r['pval']:>10.4f}")
    ai_rows.append({"feature":name,"icc":r["ICC"],"ci_lo":r["CI95%"][0],"ci_hi":r["CI95%"][1],"type":"AI"})

print(f"\n{'='*60}\n  RAW L/R VOLUMES — ICC(3,1)\n{'='*60}")
for i,col in enumerate(present_cols):
    r = _icc(col, X_bl[:,i], X_y2[:,i], ids_valid)
    print(f"{col:<35} {r['ICC']:>8.3f} [{r['CI95%'][0]:.3f},{r['CI95%'][1]:.3f}] {r['pval']:>10.4f}")
    raw_rows.append({"feature":col,"icc":r["ICC"],"ci_lo":r["CI95%"][0],"ci_hi":r["CI95%"][1],"type":"Raw"})

all_icc_df = pd.DataFrame(ai_rows + raw_rows)
plot_icc_bars(all_icc_df, save_path=fig_dir / "icc_reliability.png")

## Accelerated Maturation Visualizations

Wu et al. (2025, Nature Communications) show pallidum AI **decreases** with age as normal development (t = -7.35, p < 1e-13 in ABCD data). Our high-severity children start with less leftward pallidum, consistent with **accelerated maturation** rather than simple reduced lateralization.

1. **Offset trajectory plot** — Controls vs high-severity, same slope, different intercept
2. **Spaghetti plot** — Individual trajectories colored by severity group

In [ ]:
import numpy as np
from scipy.stats import ttest_ind
from core.regression.univariate import compute_asymmetry_features
from core.regression.longitudinal import get_paired_bl_y2
from core.regression.visualization import plot_offset_trajectories

id_col      = env.configs.data["columns"]["mapping"]["id"]
baseline_tp = env.configs.data["timepoints"]["baseline"]
year2_tp    = env.configs.data["timepoints"]["year2"]

bl_v, y2_v, X_bl, X_y2, asym_bl, asym_y2, valid_pairs, present_cols = get_paired_bl_y2(
    long_df, tp_col, baseline_tp, year2_tp, roi_columns, bilateral_pairs, id_col,
    target_col=TARGET_COL, harm_cfg=dict(env.configs.harmonize))
y_sev = bl_v[TARGET_COL].values.astype(float)

ctrl = y_sev == 0; high = y_sev >= CUTOFF; low = (y_sev>=1) & (y_sev<CUTOFF)
groups_data = [
    ("Controls (PQ-BC = 0)", ctrl, "#2196F3"),
    ("Low Severity (1-29)",  low,  "#FF9800"),
    ("High Severity (≥30)",  high, "#E53935"),
]

delta_ctrl = asym_y2["pallidum_AI"][ctrl].mean() - asym_bl["pallidum_AI"][ctrl].mean()
delta_high = asym_y2["pallidum_AI"][high].mean() - asym_bl["pallidum_AI"][high].mean()
bl_diff    = asym_bl["pallidum_AI"][ctrl].mean() - asym_bl["pallidum_AI"][high].mean()
t_bl, p_bl = ttest_ind(asym_bl["pallidum_AI"][ctrl], asym_bl["pallidum_AI"][high])

print(f"Controls Δ={delta_ctrl:.4f}  |  High Δ={delta_high:.4f}  |  BL gap={bl_diff:.4f}, p={p_bl:.3f}")
plot_offset_trajectories(groups_data, asym_bl, asym_y2, bl_diff, p_bl,
                          delta_ctrl, delta_high,
                          save_path=fig_dir / "offset_trajectories.png")

## Year 2 Cross-Sectional Replication

Two complementary analyses:
1. **Baseline-defined** (PQ-BC ≥ 30 at baseline): Tests whether baseline high-severity kids still show brain-symptom signal at Year 2. Expected to fail due to 90% symptom remission.
2. **Year 2-defined** (PQ-BC ≥ 30 at Year 2): True cross-sectional replication — same analysis as baseline, just at a different timepoint. Tests whether the brain-symptom relationship is a replicable concurrent phenomenon.

In [ ]:
# Year 2 cross-sectional replication (baseline-defined vs Year 2-defined vs Worseners)
import importlib, core.regression.longitudinal, core.regression.visualization
importlib.reload(core.regression.longitudinal)
importlib.reload(core.regression.visualization)

from core.regression.longitudinal import run_y2_replication
from core.regression.visualization import plot_y2_svr_scatter

y2_rep = run_y2_replication(long_df, bilateral_pairs, roi_columns, TARGET_COL,
                             env, CUTOFF, N_SPLITS, tp_col, include_worseners=True)

# Scatter: Y2-defined SVR
from scipy.stats import pearsonr
r_y2, p_y2 = pearsonr(y2_rep["y2def"]["all_true"], y2_rep["y2def"]["all_pred"])
plot_y2_svr_scatter(y2_rep["y2def"]["all_true"], y2_rep["y2def"]["all_pred"],
    r_y2, p_y2, label="Y2-defined SVR",
    save_path=fig_dir / "y2_replication_scatter.png")


## Year 2 Deep Dive: Within-Subject Change, AI vs Volume, Bootstrap CIs, Persistent Group

Four follow-up analyses on the Year 2 replication finding :
1. **Within-subject change**: Does change in pallidum AI track change in PQ-BC within individuals?
2. **AI vs Total volume**: Confirm asymmetry (not volume) drives the Y2 signal
3. **Bootstrap CIs**: Statistical precision on Y2 pallidum AI correlation
4. **Persistent high group**: The ~30 kids high at both timepoints


In [ ]:
# Year 2 Deep Dive: 5 Follow-Up Analyses
from core.regression.longitudinal import run_y2_deep_dive

present_cols = [c for c in roi_columns if c in long_df.columns]
valid_pairs_y2 = [(n,l,r) for n,l,r in bilateral_pairs if l in present_cols and r in present_cols]

y2_dd = run_y2_deep_dive(
    long_df, bilateral_pairs, present_cols, valid_pairs_y2,
    TARGET_COL, N_BOOTSTRAP, SEED, tp_col, env,
)
print(f"\nY2 pallidum AI: r={y2_dd['boot_result']['r_obs']:+.4f}, "
      f"95% CI=[{y2_dd['boot_result']['ci_lo']:+.4f}, {y2_dd['boot_result']['ci_hi']:+.4f}]")

In [ ]:
# Sex-stratified hemispheric decomposition at Year 2
import numpy as np
from statsmodels.stats.multitest import multipletests
from core.regression.univariate import compute_asymmetry_features
from core.regression.longitudinal import _combat_harmonize as _ch
from core.regression.visualization import plot_sex_hemi_y2
from core.regression.evaluation import fisher_z_compare
from scipy.stats import pearsonr

sex_col = env.configs.data["columns"]["mapping"]["sex_mapped"]
id_col  = env.configs.data["columns"]["mapping"]["id"]
year2_tp    = env.configs.data["timepoints"]["year2"]
baseline_tp = env.configs.data["timepoints"]["baseline"]

bl = long_df[long_df[tp_col] == baseline_tp].copy()
y2 = long_df[long_df[tp_col] == year2_tp].copy()
present_cols_y2 = [c for c in roi_columns if c in y2.columns]
valid_pairs_sex = [(n,l,r) for n,l,r in bilateral_pairs if l in present_cols_y2 and r in present_cols_y2]
pal_pair = [(n,l,r) for n,l,r in valid_pairs_sex if "pallidum" in n.lower()]
pal_name, pal_l, pal_r = pal_pair[0] if pal_pair else ("pallidum", "", "")

harm_cfg = dict(env.configs.harmonize)

# NOTE: ComBat fit on full Y2 sample (not per-fold) — appropriate for correlation analyses.
# Per-fold ComBat is used in SVR cells only.
y2_clean = y2.dropna(subset=[sex_col]).reset_index(drop=True)
X_y2_full_raw = y2_clean[present_cols_y2].values.astype(float)
X_y2_full, keep_mask, _ = _ch(X_y2_full_raw, y2_clean, harm_cfg)
y2_harm = y2_clean[keep_mask].reset_index(drop=True)
X_y2_harm = X_y2_full
print(f"Full Y2 ComBat: {len(y2_clean)} -> {len(y2_harm)} subjects")

bl_high_ids = bl[bl[TARGET_COL] >= CUTOFF][id_col].values
cohort_masks = {
    "Baseline-defined": y2_harm[id_col].isin(bl_high_ids).values,
    "Year 2-defined":   (y2_harm[TARGET_COL] >= CUTOFF).values,
}

cohort_results = {}
for label, mask_c in cohort_masks.items():
    y2_sev = y2_harm[mask_c].reset_index(drop=True)
    X_c    = X_y2_harm[mask_c]
    if pal_l and pal_r:
        valid_rows = y2_sev[[pal_l, pal_r]].notna().all(axis=1)
        X_c    = X_c[valid_rows.values]
        y2_sev = y2_sev[valid_rows].reset_index(drop=True)
    print(f"{label}: n={len(y2_sev)}")

    asym_y2 = compute_asymmetry_features(X_c, present_cols_y2, valid_pairs_sex)
    y2_sev["pallidum_AI"] = asym_y2.get("pallidum_AI", np.zeros(len(y2_sev)))
    y_col = y2_sev[TARGET_COL].values.astype(float)

    # Residualize age from y — matches BL preprocessing (prepare_harmonized_data
    # also regresses out age). Use longitudinal age (actual age at Y2).
    # Do NOT fall back to demo_brthdat_v2 (baseline age carried forward — wrong for Y2).
    from sklearn.linear_model import LinearRegression as _LR
    _age_col = "demo_brthdat_v2_l"  # longitudinal age, available at all timepoints
    if _age_col in y2_sev.columns:
        _age_v = y2_sev[_age_col].values.astype(float).reshape(-1, 1)
        _ok = np.isfinite(_age_v.ravel()) & np.isfinite(y_col)
        if _ok.sum() > 10:
            _rm = _LR().fit(_age_v[_ok], y_col[_ok])
            # Only subtract predictions where both age and y are finite;
            # avoids NaN propagation via predict(NaN) for subjects missing age.
            y_col[_ok] = y_col[_ok] - _rm.predict(_age_v[_ok])
            print(f"  [{label}] Age-residualized y using {_age_col} "
                  f"(n_residualized={_ok.sum()}, n_skipped={int((~_ok).sum())})")
    else:
        print(f"  WARNING: {_age_col} not found — y_col is raw (not age-residualized)")
    asym_y2[pal_l] = X_c[:, present_cols_y2.index(pal_l)] if pal_l in present_cols_y2 else np.zeros(len(y2_sev))
    asym_y2[pal_r] = X_c[:, present_cols_y2.index(pal_r)] if pal_r in present_cols_y2 else np.zeros(len(y2_sev))

    # Collect raw p-values for FDR correction (2 sexes x 3 features = 6 tests per cohort)
    raw_tests = []  # (sex, feat_name, r, p, n)
    for sex in ["male", "female"]:
        smask = y2_sev[sex_col] == sex
        for feat, fname in [("pallidum_AI", "AI"), (pal_l, "L"), (pal_r, "R")]:
            if feat in asym_y2 and smask.sum() > 5:
                r, p = pearsonr(asym_y2[feat][smask.values], y_col[smask])
                raw_tests.append((sex, fname, r, p, int(smask.sum())))

    # FDR correction across all tests in this cohort
    if raw_tests:
        pvals = [t[3] for t in raw_tests]
        _, p_fdr, _, _ = multipletests(pvals, method="fdr_bh")
        print(f"  {'Sex':<8} {'Feat':<6} {'r':>8} {'p_raw':>8} {'p_fdr':>8} {'n':>5}")
        print(f"  {'-'*46}")
        for (sex, fname, r, p_raw, n), p_fdr_val in zip(raw_tests, p_fdr):
            sig = "***" if p_fdr_val<0.001 else ("**" if p_fdr_val<0.01 else ("*" if p_fdr_val<0.05 else "n.s."))
            print(f"  {sex:<8} {fname:<6} {r:>+8.3f} {p_raw:>8.4f} {p_fdr_val:>8.4f} {n:>5}  {sig}")

    # Fisher z-tests: male vs female for each feature
    rs_by_sex = {}
    ns_by_sex = {}
    for sex in ["male", "female"]:
        smask = y2_sev[sex_col] == sex
        rs_by_sex[sex] = {}; ns_by_sex[sex] = {}
        for feat, fname in [("pallidum_AI", "AI"), (pal_l, "L"), (pal_r, "R")]:
            if feat in asym_y2 and smask.sum() > 5:
                r, _ = pearsonr(asym_y2[feat][smask.values], y_col[smask])
                rs_by_sex[sex][fname] = r
                ns_by_sex[sex][fname] = int(smask.sum())

    print(f"Fisher z-tests (male vs female) — {label}:")
    for fname in ["AI", "L", "R"]:
        if fname in rs_by_sex.get("male",{}) and fname in rs_by_sex.get("female",{}):
            z, p_z = fisher_z_compare(
                rs_by_sex["male"][fname], ns_by_sex["male"][fname],
                rs_by_sex["female"][fname], ns_by_sex["female"][fname],
            )
            sig = "*" if p_z < 0.05 else "n.s."
            print(f"    {fname}: z={z:+.3f}, p={p_z:.4f} {sig}  (male r={rs_by_sex['male'][fname]:+.3f}, female r={rs_by_sex['female'][fname]:+.3f})")

    cohort_results[label] = {"df": y2_sev, "y": y_col, "asym": asym_y2, "sex_col": sex_col}

plot_sex_hemi_y2(cohort_results, valid_pairs_sex, present_cols_y2, pal_name, pal_l, pal_r,
    TARGET_COL, save_path=fig_dir / "sex_hemi_y2.png")


## Sex-Stratified SVR at Year 2

Replication of NB07 sex-stratified SVR on the **Year 2-defined** high-severity sample
(Y2 PQ-BC >= 30, n=143; 79% new subjects). At baseline, the univariate pallidum AI effect
is male-specific (r=−0.25 males vs r=−0.03 females). This tests whether that sex
asymmetry replicates or inverts at Year 2.

**ComBat**: Applied on full Y2 high-severity sample (both sexes combined) before splitting
by sex — avoids unstable scanner statistics with n≈35-42 per scanner per sex per fold.

In [ ]:
# Sex-stratified SVR at Year 2 — loop over Y2-defined and Worseners cohorts
import numpy as np
from scipy.stats import pearsonr
from core.regression.longitudinal import (
    run_sex_stratified_svr, perm_and_boot_svr, build_y2_high_df, _combat_harmonize)
from core.regression.evaluation import fisher_z_compare

harm_config = env.configs.harmonize
scanner_col = harm_config.get("site_column", "mri_info_manufacturer")
sex_col_name = "sex_mapped"

_reg_covs = env.configs.regression.get("covariates", {}).get("columns", [])
_age_col_y2 = "demo_brthdat_v2_l"
cov_cols = [_age_col_y2 if c == "demo_brthdat_v2" else c for c in _reg_covs]

all_cohort_results = {}
for cohort_mode in ["y2_defined", "worseners"]:
    print(f"\n{'='*60}\n  SEX-STRATIFIED SVR: {cohort_mode.upper()}\n{'='*60}")

    y2_high, y_y2h, bil_p, val_p, pres_y2 = build_y2_high_df(
        long_df, TARGET_COL, env, CUTOFF, tp_col, roi_columns, cohort_mode=cohort_mode)

    X_raw = y2_high[pres_y2].values.astype(float)
    X_harm, keep_mask, _ = _combat_harmonize(X_raw, y2_high, harm_config, min_site_n=5)
    y2_high_h = y2_high[keep_mask].reset_index(drop=True)
    print(f"Y2 high-severity after ComBat: n={len(y2_high_h)}")

    sex_y2_results = {}
    for sex_label in ["male", "female"]:
        sex_mask = (y2_high_h[sex_col_name] == sex_label).values
        sex_df   = y2_high_h[sex_mask].copy().reset_index(drop=True)
        X_sex    = X_harm[sex_mask]
        y_sex    = y2_high_h.loc[sex_mask, TARGET_COL].values.astype(float)

        if len(sex_df) < 10:
            print(f"  {sex_label}: n={len(sex_df)} too small, skipping"); continue
        print(f"\n  {sex_label} n={len(sex_df)}")

        fold_data, all_t, all_p = run_sex_stratified_svr(
            X_sex, y_sex, sex_df, sex_label, cov_cols, sex_col_name, scanner_col,
            n_splits=N_SPLITS, seed=SEED,
        )
        boot_r, null_r, p_emp = perm_and_boot_svr(
            fold_data, all_t, all_p,
            n_perms=N_PERMS, n_boot=N_BOOTSTRAP, seed=SEED, label=sex_label,
        )
        r_all = pearsonr(all_t, all_p)[0]
        ci = np.percentile(boot_r, [2.5, 97.5])
        sex_y2_results[sex_label] = {
            "fold_data": fold_data, "all_t": all_t, "all_p": all_p,
            "r": r_all, "boot_r": boot_r, "null_r": null_r, "p_emp": p_emp,
        }
        print(f"  {sex_label}: r={r_all:.4f}, p_perm={p_emp:.4f}, 95% CI=[{ci[0]:.3f},{ci[1]:.3f}]")

    if len(sex_y2_results) == 2:
        rm = sex_y2_results["male"]["r"]; nm = len(sex_y2_results["male"]["all_t"])
        rf = sex_y2_results["female"]["r"]; nf = len(sex_y2_results["female"]["all_t"])
        z_sex, p_sex = fisher_z_compare(rm, nm, rf, nf)
        sig = "*" if p_sex < 0.05 else "n.s."
        ci_m = np.percentile(sex_y2_results["male"]["boot_r"], [2.5, 97.5])
        ci_f = np.percentile(sex_y2_results["female"]["boot_r"], [2.5, 97.5])
        print(f"Y2 SVR summary ({cohort_mode}):")
        print(f"  male:   r={rm:+.4f}, p_perm={sex_y2_results['male']['p_emp']:.4f}, 95%CI=[{ci_m[0]:+.3f},{ci_m[1]:+.3f}] (n={nm})")
        print(f"  female: r={rf:+.4f}, p_perm={sex_y2_results['female']['p_emp']:.4f}, 95%CI=[{ci_f[0]:+.3f},{ci_f[1]:+.3f}] (n={nf})")
        print(f"  Fisher z-test (male vs female): z={z_sex:+.3f}, p={p_sex:.4f} {sig}")

    all_cohort_results[cohort_mode] = sex_y2_results


## Prospective Prediction: Baseline Brain → Year 2 Severity

Take subjects who are **high-severity at Year 2** (Y2 PQ-BC ≥ 30, n≈143),
then use their **baseline brain** (age 9-10) asymmetry features to predict
their Year 2 PQ-BC severity.

This is a prospective test: 79% of the Y2 high-severity group are new-onset
(BL PQ-BC < 30), so baseline brain predicting Year 2 severity in this group
is genuine forward prediction, not concurrent association.

Compares directly against the concurrent result (Y2 brain → Y2 PQ-BC,
already shown: pallidum AI r=−0.194, p=0.021).

In [ ]:
# Prospective SVR: Baseline brain → Year 2 severity — loop over cohort modes
from core.regression.longitudinal import run_prospective_svr
from core.regression.visualization import plot_y2_svr_scatter
from scipy.stats import pearsonr

for cohort_mode in ["y2_defined", "worseners"]:
    print(f"\n{'='*60}\n  PROSPECTIVE SVR: {cohort_mode.upper()}\n{'='*60}")
    prosp_res = run_prospective_svr(
        long_df, bilateral_pairs, TARGET_COL, env, CUTOFF, N_SPLITS, N_PERMS,
        N_BOOTSTRAP, SEED, tp_col, cohort_mode=cohort_mode)

    print(f"\nProspective r = {prosp_res['r']:.4f}  p_perm = {prosp_res['p_perm']:.4f}")
    print(f"Bootstrap 95% CI: [{prosp_res['boot_ci'][0]:.4f}, {prosp_res['boot_ci'][1]:.4f}]")
    print(f"n = {prosp_res['n']}")

    plot_y2_svr_scatter(prosp_res["all_true"], prosp_res["all_pred"],
        prosp_res["r"], prosp_res["p_perm"],
        label=f"Prospective BL→Y2 SVR ({cohort_mode}, n={prosp_res['n']})",
        save_path=fig_dir / f"prospective_svr_scatter_{cohort_mode}.png")


## Symptom Regression & Persistent-Symptom SVR

1. **Symptom regression check**: How much did PQ-BC scores drop from baseline to Year 2?
2. **Persistent-symptom SVR**: Restrict to subjects whose Year 2 PQ-BC remained ≥ 30 — does the brain signal survive?

In [ ]:
# Worsening group: baseline brain → concurrent Year 2 SVR
# Clinically-worsened: delta PQ-BC >= 1 SD (MCID threshold)
from core.regression.longitudinal import run_worsening_analysis
from scipy.stats import pearsonr

worsen_res = run_worsening_analysis(
    long_df, bilateral_pairs, TARGET_COL, env, N_SPLITS, N_PERMS, SEED, tp_col)

r_y2 = worsen_res["res_y2"]["r"]; r_bl = worsen_res["res_bl"]["r"]
print(f"\nMCID = {worsen_res['mcid']:.0f} | N worsened = {worsen_res['n_worsened']}")
print(f"Y2 concurrent SVR: r = {r_y2:.4f}")
print(f"BL prospective SVR: r = {r_bl:.4f}")

In [ ]:
## Analysis 1: Baseline Brain → ΔPQ-BC (Continuous SVR, Full Paired Sample)
from core.regression.longitudinal import run_delta_svr
from scipy.stats import pearsonr

delta_res = run_delta_svr(long_df, bilateral_pairs, TARGET_COL, env, N_SPLITS, SEED, tp_col)
r_d = delta_res["r"]; p_d = delta_res["p"]; n_d = delta_res["n"]
print(f"\nDelta SVR (BL brain → ΔPQ-BC): r={r_d:+.4f}, p={p_d:.4f}, n={n_d}")

In [ ]:
## Analysis 2: Baseline Brain → Worsened/Stable Multi-Algo Classifier (MCID = 0.5 SD)
import importlib, core.regression.longitudinal
importlib.reload(core.regression.longitudinal)

from core.regression.longitudinal import run_multi_algo_worsening_classifier

multi_res = run_multi_algo_worsening_classifier(
    long_df, bilateral_pairs, TARGET_COL, env, N_SPLITS, SEED, tp_col, mcid_factor=0.5)

print(f"\nWorsening Classifier Comparison (0.5 SD threshold):")
print(f"  n_worsened={multi_res['n_worsened']}, n_stable={multi_res['n_stable']}")
print(multi_res["comparison_df"].to_string(index=False))


## Developmental Asymmetry: Quantification & 3-Timepoint Trajectory

This section quantifies:

1. **Baseline AI gap** between controls and high-severity (permutation test + bootstrap CIs)
2. **3-timepoint trajectory** — does the gap persist, converge, or diverge from BL -> Y2 -> Y4?

**NOTE**: Developmental age offset computation (dividing observed gap by normative decline rate from
Wu et al. 2025) was removed. Applying a rate from a different study to convert a group difference
into a "years of offset" is methodologically unsound without matched methodology.

In [ ]:
# Developmental asymmetry quantification + 3-timepoint trajectory
from core.regression.longitudinal import run_asymmetry_quantification

aq_res = run_asymmetry_quantification(
    long_df, bilateral_pairs, TARGET_COL, env, N_PERMS, N_BOOTSTRAP, SEED, CUTOFF, tp_col)

gap = aq_res["gap_result"]
print(f"\nSummary:")
print(f"  BL AI gap: {gap['observed']:.5f}, p_perm={gap['p_perm']:.4f}, CI=[{gap['ci'][0]:.5f}, {gap['ci'][1]:.5f}]")
print(f"\nDevelopmental change BL→Y2 (all subjects):")
for ai_name, dr in aq_res["dev_results"].items():
    sig = "*" if dr["p"] < 0.05 else ""
    print(f"  {ai_name}: BL={dr['bl_mean']:+.4f} → Y2={dr['y2_mean']:+.4f} (p={dr['p']:.4f}{sig})")

## Persistent Cases: BL ≥ 30 AND Y2 ≥ 30

Among baseline high-severity subjects (PQ-BC ≥ 30, n≈476), the vast majority
remit by Year 2 (~90%). This cell isolates the **persistent** subgroup (BL ≥ 30
**AND** Y2 ≥ 30) and compares their baseline brain asymmetry to **remitted** subjects
(BL ≥ 30 AND Y2 < 30).

If asymmetry is a trait marker, persistent cases should show **more extreme** pallidum AI
at baseline — and the pattern should be maintained at Year 2.

Key groups:
- **Persistent**: BL ≥ 30 AND Y2 ≥ 30 (n≈33)
- **Remitted**: BL ≥ 30 AND Y2 < 30 (n≈277)

In [ ]:
# Persistent vs Remitted: Baseline Brain Asymmetry Comparison
from core.regression.longitudinal import run_persistent_remitted
from core.regression.visualization import plot_persistent_remitted

pr_res = run_persistent_remitted(long_df, bilateral_pairs, TARGET_COL, env, CUTOFF, N_BOOTSTRAP, SEED, tp_col)

plot_persistent_remitted(
    pr_res["ai_names"], pr_res["asym"], pr_res["mask_p"], pr_res["mask_r"],
    pr_res["n_persistent"], pr_res["n_remitted"],
    pr_res["bl_matched"], pr_res["y2_matched"], TARGET_COL, CUTOFF,
    save_path=fig_dir / "persistent_remitted.png")